# Salary Prediction — Dataset Producer (`developer_events`) — VM run

Runs `src/producers/dataset_producer.py` (PLAN.md §14 / §24 Phase 6) from this notebook and verifies events land on `developer_events` — no separate terminal needed.

Unlike `run_prediction_stream.ipynb`, **this notebook does not need Spark or the Kafka connector JAR at all** — the producer is plain Python (`csv` + `confluent_kafka`), so any Jupyter session works, not just one launched via `scripts/start_kafka_jupyter.sh`.

**Prerequisites:** Kafka broker running, `developer_events` topic created, dataset CSV present at `settings.DATASET_PATH`. **Run cells top to bottom.**

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

If a later cell fails with `ModuleNotFoundError`, add `%pip install <package-name>` here and re-run from the top after restarting the kernel.

In [ ]:
%pip install python-dotenv confluent-kafka

## 2. `.env` check

In [ ]:
if not os.path.exists(".env"):
    import subprocess
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")

print(open(".env").read())

Confirm `KAFKA_BOOTSTRAP_SERVERS`, `KAFKA_DATASET_TOPIC`, and `DATASET_PATH` above match what you actually created on the VM (`developer_events`, per PLAN.md §6 / §8).

In [ ]:
from config import settings
from src.producers.dataset_producer import stream_dataset

print("KAFKA_BOOTSTRAP_SERVERS      =", settings.KAFKA_BOOTSTRAP_SERVERS)
print("KAFKA_DATASET_TOPIC          =", settings.KAFKA_DATASET_TOPIC)
print("DATASET_PATH                 =", settings.DATASET_PATH)
print("DATASET_PATH exists?         =", os.path.exists(settings.DATASET_PATH))
print("DATASET_EVENT_DELAY_SECONDS  =", settings.DATASET_EVENT_DELAY_SECONDS)

## 3. Publish a bounded batch of events

The full survey CSV is ~89K rows — at any reasonable demo delay that would take hours to replay. This cell publishes just `LIMIT` events (default 20) with a short delay, enough to prove the producer works end-to-end. Raise `LIMIT` or drop `DELAY_SECONDS` if you want a bigger/faster batch; see the bottom of this notebook for an unbounded, full-file replay.

In [ ]:
LIMIT = 20
DELAY_SECONDS = 0.5

published = stream_dataset(
    csv_path=settings.DATASET_PATH,
    topic=settings.KAFKA_DATASET_TOPIC,
    delay_seconds=DELAY_SECONDS,
    limit=LIMIT,
)
print("Published", published, "events to", settings.KAFKA_DATASET_TOPIC)

## 4. Verify events landed on `developer_events`

Uses a plain `confluent_kafka` consumer (not Spark) to read back whatever is currently on the topic, from the beginning, for a few seconds.

In [ ]:
import json
import time

from confluent_kafka import Consumer

consumer = Consumer({
    "bootstrap.servers": settings.KAFKA_BOOTSTRAP_SERVERS,
    "group.id": "run-dataset-producer-notebook-verify",
    "auto.offset.reset": "earliest",
})
consumer.subscribe([settings.KAFKA_DATASET_TOPIC])

messages = []
deadline = time.time() + 10
while time.time() < deadline:
    msg = consumer.poll(timeout=1.0)
    if msg is None:
        continue
    if msg.error():
        print("Consumer error:", msg.error())
        continue
    messages.append(json.loads(msg.value()))

consumer.close()

print(f"Read {len(messages)} event(s) currently on {settings.KAFKA_DATASET_TOPIC}.")
if messages:
    print("Most recent event:")
    print(json.dumps(messages[-1], indent=2))

## 5. (Optional) Full, unbounded replay

To replay the *entire* CSV (not just `LIMIT` rows), run the cell below instead. With the default `DATASET_EVENT_DELAY_SECONDS` this takes a long time (89,184 rows × delay) — lower the delay if you just want to see it run to completion faster. Interrupt the kernel (Kernel → Interrupt) to stop early; the producer shuts down gracefully and flushes whatever's already queued.

In [ ]:
# Uncomment to run a full, unbounded replay of the dataset.
# published = stream_dataset(
#     csv_path=settings.DATASET_PATH,
#     topic=settings.KAFKA_DATASET_TOPIC,
#     delay_seconds=settings.DATASET_EVENT_DELAY_SECONDS,
# )
# print("Published", published, "events to", settings.KAFKA_DATASET_TOPIC)